In [1]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import os

In [2]:
df_tickers = pd.read_csv("DIM_Tickers.csv")
lista_tickers = df_tickers["TickerYahoo"].drop_duplicates().tolist()

print(f"Tickers a procesar: {len(lista_tickers)}")
print(lista_tickers)

Tickers a procesar: 38
['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA', 'ORCL', 'PLTR', 'RDDT', 'MU', 'AMD', 'TSM', 'ALAB', 'MSTR', 'QCOM', 'YPF', 'PEP', 'UNH', 'VIST', 'SPY', 'QQQ', 'DIA', 'IWM', 'EEM', 'VEA', 'XLK', 'XLE', 'XLF', 'XLV', 'GLD', 'SLV', 'USO', 'EWZ', 'URA', 'SOXX', 'BTC-USD', 'ETH-USD']


In [3]:
CSV_PATH = "precios_historicos.csv"

if os.path.exists(CSV_PATH):
    df_existente = pd.read_csv(CSV_PATH)
    # format='mixed' maneja tanto "2026-04-28" como "2026-04-28 00:00:00"
    df_existente["Date"] = pd.to_datetime(df_existente["Date"], format="mixed")
    ultima_fecha = df_existente["Date"].max()
    fecha_inicio = (ultima_fecha + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    fecha_fin    = datetime.today().strftime("%Y-%m-%d")
    MODO = "incremental"
    print(f"Modo: INCREMENTAL — desde {fecha_inicio} hasta {fecha_fin}")
    print(f"Registros existentes: {len(df_existente):,}")
else:
    df_existente = None
    fecha_inicio = None
    fecha_fin    = None
    MODO = "completo"
    print("Modo: COMPLETO — descargando historial de 5 años")

Modo: INCREMENTAL — desde 2026-04-30 hasta 2026-05-01
Registros existentes: 47,487


In [4]:
lista_precios = []

for t in lista_tickers:
    try:
        tk = yf.Ticker(t)

        if MODO == "incremental":
            hist = tk.history(start=fecha_inicio, end=fecha_fin)
        else:
            hist = tk.history(period="5y")

        if hist.empty:
            print(f"Sin datos nuevos para {t}")
            continue

        hist = hist.reset_index()
        hist["Ticker"] = t
        hist = hist[["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"]]
        # Normalizamos fecha a string YYYY-MM-DD para evitar conflictos de timezone
        hist["Date"] = pd.to_datetime(hist["Date"]).dt.tz_localize(None)

        lista_precios.append(hist)
        print(f"OK: {t} — {len(hist)} registros nuevos")

    except Exception as e:
        print(f"Error en {t}: {e}")

print(f"\nTickers procesados: {len(lista_precios)} de {len(lista_tickers)}")

OK: AAPL — 1 registros nuevos
OK: MSFT — 1 registros nuevos
OK: GOOGL — 1 registros nuevos
OK: AMZN — 1 registros nuevos
OK: META — 1 registros nuevos
OK: NVDA — 1 registros nuevos
OK: TSLA — 1 registros nuevos
OK: ORCL — 1 registros nuevos
OK: PLTR — 1 registros nuevos
OK: RDDT — 1 registros nuevos
OK: MU — 1 registros nuevos
OK: AMD — 1 registros nuevos
OK: TSM — 1 registros nuevos
OK: ALAB — 1 registros nuevos
OK: MSTR — 1 registros nuevos
OK: QCOM — 1 registros nuevos
OK: YPF — 1 registros nuevos
OK: PEP — 1 registros nuevos
OK: UNH — 1 registros nuevos
OK: VIST — 1 registros nuevos
OK: SPY — 1 registros nuevos
OK: QQQ — 1 registros nuevos
OK: DIA — 1 registros nuevos
OK: IWM — 1 registros nuevos
OK: EEM — 1 registros nuevos
OK: VEA — 1 registros nuevos
OK: XLK — 1 registros nuevos
OK: XLE — 1 registros nuevos
OK: XLF — 1 registros nuevos
OK: XLV — 1 registros nuevos
OK: GLD — 1 registros nuevos
OK: SLV — 1 registros nuevos
OK: USO — 1 registros nuevos
OK: EWZ — 1 registros nuevos


In [5]:
if lista_precios:
    df_nuevos = pd.concat(lista_precios, ignore_index=True)

    if MODO == "incremental" and df_existente is not None:
        df_final = pd.concat([df_existente, df_nuevos], ignore_index=True)
        df_final["Date"] = pd.to_datetime(df_final["Date"], format="mixed")
        df_final = df_final.drop_duplicates(subset=["Date", "Ticker"]).reset_index(drop=True)
    else:
        df_final = df_nuevos

    # Guardamos la fecha como string YYYY-MM-DD para evitar problemas en Power BI
    df_final["Date"] = pd.to_datetime(df_final["Date"], format="mixed").dt.strftime("%Y-%m-%d")

    df_final.to_csv(CSV_PATH, index=False)
    print(f"Archivo guardado: {CSV_PATH}")
    print(f"Total registros: {len(df_final):,}")
    print(f"Muestra de fechas: {df_final['Date'].head(3).tolist()}")
else:
    print("No hay datos nuevos — CSV sin cambios")

Archivo guardado: precios_historicos.csv
Total registros: 47,525
Muestra de fechas: ['2021-04-27', '2021-04-28', '2021-04-29']
